In [1]:
from pyspark.sql import(
    functions as f, 
    SparkSession, 
    types as t
)

In [2]:
spark = SparkSession.builder.appName('df_most_interviewed').getOrCreate()

In [3]:
table_schema = t.StructType([
    t.StructField("interviwer_id", t.StringType(), False),
    t.StructField("occupation_id", t.StringType(), False),
    t.StructField("rating", t.IntegerType(), False)])

In [7]:
csv_file_path = "file:///home/jovyan/work/sample/like.csv"
df = spark.read.schema(table_schema).csv(csv_file_path)
df.show()

+-------------+-------------+------+
|interviwer_id|occupation_id|rating|
+-------------+-------------+------+
|        11657|         1100|     8|
|        13727|         2030|     2|
|        59892|         3801|     1|
|         6538|         3021|     6|
|        95811|         2030|     9|
|        54500|         1100|    10|
|        69741|         2030|     3|
|        51166|         2030|    10|
|        70009|         9382|     5|
|        63152|         2030|     6|
|        70758|         1100|     2|
|        35580|         2030|     5|
|        63199|         1100|    10|
|        33078|         2030|     3|
|        97480|         9382|     2|
|        47223|         1100|     8|
|        80308|         3021|     8|
|        26691|         1100|     3|
|        17194|         3021|     3|
|        96584|         2030|     4|
+-------------+-------------+------+
only showing top 20 rows



In [9]:
interviwer_count = df.groupBy('occupation_id').count().orderBy(f.desc("count"))
interviwer_count.show()

+-------------+-----+
|occupation_id|count|
+-------------+-----+
|         1100|  217|
|         3801|  203|
|         2030|  200|
|         3021|  191|
|         9382|  189|
+-------------+-----+



In [10]:
meta = {
    '1100' : 'engineer', 
    '2030' : 'devloper', 
    '3801' : 'painter', 
    '3021' : 'chemistry teacher', 
    '9382' : 'priest'
}

In [11]:
occupation_dict = spark.sparkContext.broadcast(meta)

In [ ]:
def get_occupation_name(occupation_id: str) -> str:
    return occupation_dict.value[occupation_id]

In [14]:
occupation_lookup_udf = f.udf(get_occupation_name)

In [17]:
occupation_with_name = interviwer_count.withColumn('occupation_name', occupation_lookup_udf(f.col("occupation_id")))
occupation_with_name.show(10)

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/tmp/ipykernel_2205/1987664404.py", line 2, in get_occupation_name
AttributeError: 'str' object has no attribute 'value'
